In [18]:
# Tìm thư mục gốc để notebook chạy được từ notebooks/ hoặc thư mục project.
from pathlib import Path
import sys

project_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'src').is_dir()
)
sys.path.insert(0, str(project_root))
DATA_DIR = project_root / 'data'
MODEL_DIR = project_root / 'models'
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import classification_report

In [19]:
from src.models.fine_tuning import build_model, fine_tune_model

In [20]:
train_df = pd.read_csv(DATA_DIR / 'train_split.csv')
val_df = pd.read_csv(DATA_DIR / 'validation_split.csv')
test_df = pd.read_csv(DATA_DIR / 'test_split.csv')

# Ảnh được đổi về cùng kích thước và chuẩn hóa giống pipeline hiện tại.

def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.io.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, (160, 160))

    # Chuẩn hóa ảnh về [0, 1]
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

def make_dataset(frame, training=False):
    paths = [str(DATA_DIR / name) for name in frame['file_name']]
    labels = frame['label'].to_numpy(dtype='float32')
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        dataset = dataset.shuffle(1000)
    dataset = dataset.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    return dataset.batch(16).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)
y_test = test_df['label'].to_numpy()

In [21]:
model = build_model()

# Pha 1: backbone đang đóng băng, chỉ học bộ phân loại mới.
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_auc', mode='max', patience=2, restore_best_weights=True
)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.2, patience=1, min_lr=1e-6
)
head_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[early_stop, reduce_lr],
    shuffle=False,
)

Epoch 1/15
438/438 ━━━━━━━━━━━━━━━━━━━━ 35s 76ms/step - accuracy: 0.5544 - auc: 0.5729 - loss: 0.8148 - val_accuracy: 0.6553 - val_auc: 0.7134 - val_loss: 0.6269 - learning_rate: 1.0000e-04
Epoch 2/15
438/438 ━━━━━━━━━━━━━━━━━━━━ 30s 70ms/step - accuracy: 0.6714 - auc: 0.7317 - loss: 0.6304 - val_accuracy: 0.7387 - val_auc: 0.8266 - val_loss: 0.5138 - learning_rate: 1.0000e-04
Epoch 3/15
438/438 ━━━━━━━━━━━━━━━━━━━━ 36s 81ms/step - accuracy: 0.7339 - auc: 0.8090 - loss: 0.5378 - val_accuracy: 0.7680 - val_auc: 0.8726 - val_loss: 0.4624 - learning_rate: 1.0000e-04
Epoch 4/15
438/438 ━━━━━━━━━━━━━━━━━━━━ 89s 204ms/step - accuracy: 0.7676 - auc: 0.8472 - loss: 0.4865 - val_accuracy: 0.7993 - val_auc: 0.8958 - val_loss: 0.4225 - learning_rate: 1.0000e-04
Epoch 5/15
438/438 ━━━━━━━━━━━━━━━━━━━━ 77s 175ms/step - accuracy: 0.7796 - auc: 0.8647 - loss: 0.4606 - val_accuracy: 0.8193 - val_auc: 0.9101 - val_loss: 0.3955 - learning_rate: 1.0000e-04
Epoch 6/15
438/438 ━━━━━━━━━━━━━━━━━━━━ 101s 218

In [28]:

# Pha 2: Fine-tuning MobileNet
# Mở khóa từ layer convolution cuối cùng
model = fine_tune_model(
    model,
    unfreeze_from="conv_pw_13",
    learning_rate=1e-5
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_auc",
    mode="max",
    patience=2,
    restore_best_weights=True
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=1,
    min_lr=1e-7
)

fine_tune_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[early_stop, reduce_lr],
    shuffle=False,
)


Epoch 1/15


438/438 ━━━━━━━━━━━━━━━━━━━━ 42s 93ms/step - accuracy: 0.8071 - auc: 0.9190 - loss: 0.4238 - val_accuracy: 0.8867 - val_auc: 0.9541 - val_loss: 0.2852 - learning_rate: 1.0000e-05
Epoch 2/15
438/438 ━━━━━━━━━━━━━━━━━━━━ 38s 87ms/step - accuracy: 0.8469 - auc: 0.9267 - loss: 0.3511 - val_accuracy: 0.9013 - val_auc: 0.9590 - val_loss: 0.2635 - learning_rate: 1.0000e-05
Epoch 3/15
438/438 ━━━━━━━━━━━━━━━━━━━━ 37s 84ms/step - accuracy: 0.8609 - auc: 0.9366 - loss: 0.3226 - val_accuracy: 0.9073 - val_auc: 0.9619 - val_loss: 0.2526 - learning_rate: 1.0000e-05
Epoch 4/15
438/438 ━━━━━━━━━━━━━━━━━━━━ 42s 95ms/step - accuracy: 0.8729 - auc: 0.9419 - loss: 0.3094 - val_accuracy: 0.9067 - val_auc: 0.9635 - val_loss: 0.2491 - learning_rate: 1.0000e-05
Epoch 5/15
438/438 ━━━━━━━━━━━━━━━━━━━━ 35s 81ms/step - accuracy: 0.8759 - auc: 0.9489 - loss: 0.2893 - val_accuracy: 0.9133 - val_auc: 0.9652 - val_loss: 0.2410 - learning_rate: 1.0000e-05
Epoch 6/15
438/438 ━━━━━━━━━━━━━━━━━━━━ 36s 83ms/step - accur

In [38]:
# Chỉ dùng tập test sau khi đã hoàn tất huấn luyện.
test_loss, test_accuracy, test_auc = model.evaluate(test_ds)
print(f'Test accuracy: {test_accuracy:.4f}; AUC: {test_auc:.4f}')

probabilities = model.predict(test_ds, verbose=1).ravel()
predictions = (probabilities >= 0.5).astype('int32')
print(classification_report(
    y_test, predictions, target_names=['Real', 'AI-generated'], digits=4
))

MODEL_DIR.mkdir(parents=True, exist_ok=True)
model.save(MODEL_DIR / 'fine_tuned_mobilenet.keras')

94/94 ━━━━━━━━━━━━━━━━━━━━ 4s 39ms/step - accuracy: 0.8980 - auc: 0.9679 - loss: 0.2306
Test accuracy: 0.8980; AUC: 0.9679
94/94 ━━━━━━━━━━━━━━━━━━━━ 4s 39ms/step
              precision    recall  f1-score   support

        Real     0.8922    0.9053    0.8987       750
AI-generated     0.9039    0.8907    0.8972       750

    accuracy                         0.8980      1500
   macro avg     0.8981    0.8980    0.8980      1500
weighted avg     0.8981    0.8980    0.8980      1500



In [30]:

# Lấy MobileNet backbone từ model của bạn
backbone = model.get_layer("mobilenet_backbone")

# In toàn bộ các layer bên trong MobileNet
for i, layer in enumerate(backbone.layers):
    print(i, layer.name, layer.__class__.__name__, layer.output.shape)


0 input_layer_3 InputLayer (None, 160, 160, 3)
1 conv1 Conv2D (None, 80, 80, 32)
2 conv1_bn BatchNormalization (None, 80, 80, 32)
3 conv1_relu ReLU (None, 80, 80, 32)
4 conv_dw_1 DepthwiseConv2D (None, 80, 80, 32)
5 conv_dw_1_bn BatchNormalization (None, 80, 80, 32)
6 conv_dw_1_relu ReLU (None, 80, 80, 32)
7 conv_pw_1 Conv2D (None, 80, 80, 64)
8 conv_pw_1_bn BatchNormalization (None, 80, 80, 64)
9 conv_pw_1_relu ReLU (None, 80, 80, 64)
10 conv_pad_2 ZeroPadding2D (None, 81, 81, 64)
11 conv_dw_2 DepthwiseConv2D (None, 40, 40, 64)
12 conv_dw_2_bn BatchNormalization (None, 40, 40, 64)
13 conv_dw_2_relu ReLU (None, 40, 40, 64)
14 conv_pw_2 Conv2D (None, 40, 40, 128)
15 conv_pw_2_bn BatchNormalization (None, 40, 40, 128)
16 conv_pw_2_relu ReLU (None, 40, 40, 128)
17 conv_dw_3 DepthwiseConv2D (None, 40, 40, 128)
18 conv_dw_3_bn BatchNormalization (None, 40, 40, 128)
19 conv_dw_3_relu ReLU (None, 40, 40, 128)
20 conv_pw_3 Conv2D (None, 40, 40, 128)
21 conv_pw_3_bn BatchNormalization (None, 40

In [31]:

backbone = model.get_layer("mobilenet_backbone")
for layer in backbone.layers:
    if isinstance(layer, tf.keras.layers.Conv2D):
        if layer.strides != (1, 1):
            print(
                layer.name,
                "strides =", layer.strides,
                "output =", layer.output.shape
            )
    elif isinstance(layer, tf.keras.layers.DepthwiseConv2D):
        if layer.strides != (1, 1):
            print(
                layer.name,
                "strides =", layer.strides,
                "output =", layer.output.shape
            )
print("=== CONVOLUTION LAYERS ===")
for i, layer in enumerate(backbone.layers):
    if isinstance(layer, (
        tf.keras.layers.Conv2D,
        tf.keras.layers.DepthwiseConv2D
    )):
        print(
            f"{i:3} | {layer.name:20} | "
            f"{layer.__class__.__name__:18} | "
            f"Output: {layer.output.shape}"
        )

print("\n=== POOLING LAYERS ===")
for i, layer in enumerate(backbone.layers):
    if isinstance(layer, (
        tf.keras.layers.MaxPooling2D,
        tf.keras.layers.AveragePooling2D,
        tf.keras.layers.GlobalAveragePooling2D
    )):
        print(i, layer.name, layer.__class__.__name__)



conv1 strides = (2, 2) output = (None, 80, 80, 32)
conv_dw_2 strides = (2, 2) output = (None, 40, 40, 64)
conv_dw_4 strides = (2, 2) output = (None, 20, 20, 128)
conv_dw_6 strides = (2, 2) output = (None, 10, 10, 256)
conv_dw_12 strides = (2, 2) output = (None, 5, 5, 512)
=== CONVOLUTION LAYERS ===
  1 | conv1                | Conv2D             | Output: (None, 80, 80, 32)
  4 | conv_dw_1            | DepthwiseConv2D    | Output: (None, 80, 80, 32)
  7 | conv_pw_1            | Conv2D             | Output: (None, 80, 80, 64)
 11 | conv_dw_2            | DepthwiseConv2D    | Output: (None, 40, 40, 64)
 14 | conv_pw_2            | Conv2D             | Output: (None, 40, 40, 128)
 17 | conv_dw_3            | DepthwiseConv2D    | Output: (None, 40, 40, 128)
 20 | conv_pw_3            | Conv2D             | Output: (None, 40, 40, 128)
 24 | conv_dw_4            | DepthwiseConv2D    | Output: (None, 20, 20, 128)
 27 | conv_pw_4            | Conv2D             | Output: (None, 20, 20, 256)
 3

In [36]:
model.summary()

Model: "Fine_Tuning_MobileNet"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling_1 (Rescaling)         │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenet_backbone (Functional) │ (None, 5, 5, 1024)     │     3,228,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │         1,025 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,333,189 (20.34 MB)

 Trainable params: 1,051,649 (4.01 MB)

 Non-trainable params: 2,178,240 (8.31 MB)

 Optimizer params: 2,103,300 (8.02 MB)

In [37]:

import tensorflow as tf

# Lấy backbone MobileNet
backbone = model.get_layer("mobilenet_backbone")

print("=== BACKBONE LAYER TRAINABILITY ===")

for i, layer in enumerate(backbone.layers):
    status = "TRAINABLE" if layer.trainable else "NON-TRAINABLE"
    print(f"{i:3} | {layer.name:25} | {status}")

# Đếm số layer
trainable_count = sum(layer.trainable for layer in backbone.layers)
frozen_count = len(backbone.layers) - trainable_count

print("\n=== SUMMARY ===")
print("Total layers:", len(backbone.layers))
print("Trainable layers:", trainable_count)
print("Non-trainable layers:", frozen_count)


=== BACKBONE LAYER TRAINABILITY ===
  0 | input_layer_3             | NON-TRAINABLE
  1 | conv1                     | NON-TRAINABLE
  2 | conv1_bn                  | NON-TRAINABLE
  3 | conv1_relu                | NON-TRAINABLE
  4 | conv_dw_1                 | NON-TRAINABLE
  5 | conv_dw_1_bn              | NON-TRAINABLE
  6 | conv_dw_1_relu            | NON-TRAINABLE
  7 | conv_pw_1                 | NON-TRAINABLE
  8 | conv_pw_1_bn              | NON-TRAINABLE
  9 | conv_pw_1_relu            | NON-TRAINABLE
 10 | conv_pad_2                | NON-TRAINABLE
 11 | conv_dw_2                 | NON-TRAINABLE
 12 | conv_dw_2_bn              | NON-TRAINABLE
 13 | conv_dw_2_relu            | NON-TRAINABLE
 14 | conv_pw_2                 | NON-TRAINABLE
 15 | conv_pw_2_bn              | NON-TRAINABLE
 16 | conv_pw_2_relu            | NON-TRAINABLE
 17 | conv_dw_3                 | NON-TRAINABLE
 18 | conv_dw_3_bn              | NON-TRAINABLE
 19 | conv_dw_3_relu            | NON-TRAINABLE
 20 